# Automation for file Rearranging and Compiling

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
from utility import read_and_write
import spe_xps_reader


# Meta data reading

### Setup the top path

In [2]:
# Resovle script directory dynamically
# Assume the program is a python scirpt first, and then jupyter notebook if it fails (just for migration in the future)
try:
    path_script = Path(__file__).resolve().parent
except:
    path_script = Path.cwd()
path_script

intermediary = 'example_data'
path_data = path_script / intermediary

### Read file path for meta data

In [3]:
df_md = read_and_write.read_spectrum_meta_data(path_data)
print(df_md.info())
df_md.head()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   path      4 non-null      object        
 1   wafer_id  4 non-null      string        
 2   lot_id    4 non-null      string        
 3   datetime  4 non-null      datetime64[us]
 4   tool      4 non-null      string        
dtypes: datetime64[us](1), object(1), string(3)
memory usage: 292.0+ bytes
None


,path,wafer_id,lot_id,datetime,tool
0,d:\gitlab\GGRD\Automation\example_data\K1\RawD...,NTA000003.22,NTA000003.0H,2026-05-21 13:03:47,K1
1,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,NTA000004.23,NTA000004.0H,2026-05-17 15:26:21,H1
2,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.16,NTA000001.0K,2026-05-17 01:51:17,J4
3,d:\gitlab\GGRD\Automation\example_data\K4\RawD...,NTA000002.19,NTA000002.0H,2026-05-29 02:30:46,K4


### Read T7 data and combine with meta data

In [4]:
path_part = path_data/'part6.csv'
df_md = read_and_write.meta_data_create_T7(df_md, path_part)
df_md.head()

,path,wafer_id,lot_id,datetime,tool,T7_code
0,d:\gitlab\GGRD\Automation\example_data\K1\RawD...,NTA000003.22,NTA000003.0H,2026-05-21 13:03:47,K1,A12345
1,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,NTA000004.23,NTA000004.0H,2026-05-17 15:26:21,H1,A12345
2,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.16,NTA000001.0K,2026-05-17 01:51:17,J4,A55555
3,d:\gitlab\GGRD\Automation\example_data\K4\RawD...,NTA000002.19,NTA000002.0H,2026-05-29 02:30:46,K4,A55555


### Find the nearest spectrum for all source/target tool

In [5]:
df_pair = read_and_write.create_paired_source_target_DataFrame(df_md, time_tolerance_in_hours=11112, filter_out_non_paired_data=True)
df_pair.head()

,path,wafer_id,lot_id,datetime,tool,T7_code,tool_target,wafer_id_target,path_target,datetime_difference,datetime_target
0,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.16,NTA000001.0K,2026-05-17 01:51:17,J4,A55555,K4,NTA000002.19,d:\gitlab\GGRD\Automation\example_data\K4\RawD...,12 days 00:39:29,2026-05-29 02:30:46
1,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,NTA000004.23,NTA000004.0H,2026-05-17 15:26:21,H1,A12345,K1,NTA000003.22,d:\gitlab\GGRD\Automation\example_data\K1\RawD...,3 days 21:37:26,2026-05-21 13:03:47
2,d:\gitlab\GGRD\Automation\example_data\K1\RawD...,NTA000003.22,NTA000003.0H,2026-05-21 13:03:47,K1,A12345,H1,NTA000004.23,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,3 days 21:37:26,2026-05-17 15:26:21
3,d:\gitlab\GGRD\Automation\example_data\K4\RawD...,NTA000002.19,NTA000002.0H,2026-05-29 02:30:46,K4,A55555,J4,NTA000001.16,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,12 days 00:39:29,2026-05-17 01:51:17


# Read SPE files

In [7]:
spectra = read_and_write.SPE_file_reader(df_md, use_custom=True)

### Exact Binary Bit search

In [12]:
import numpy as np

file_path = "your_file.spe"

# Define the exact byte marker that separates text from binary
# (For PHI/MultiPak files, '$$END' is the standard end-of-header marker)
HEADER_END_MARKER = b"$$END"

with open(file_path, "rb") as f:
    file_bytes = f.read()

# Find the exact byte index where the marker ends
marker_index = file_bytes.find(HEADER_END_MARKER)

if marker_index == -1:
    raise ValueError("Could not find the end of the header marker in the file.")

# Calculate the exact boundary where binary data starts
# (We add the length of the marker to skip past it)
binary_start_offset = marker_index + len(HEADER_END_MARKER)

# 1. Safely extract and decode ONLY the text header portion
header_bytes = file_bytes[:binary_start_offset]
header_text = header_bytes.decode('utf-8', errors='ignore')

print("--- SAFE HEADER METADATA ---")
print(header_text[-200:]) # Print the last 200 characters of the header to verify

# 2. Extract ONLY the binary portion
binary_bytes = file_bytes[binary_start_offset:]

# Convert the raw byte block into a NumPy array
# (PHI files typically use float32 or float64 for counts/intensities)
binary_data = np.frombuffer(binary_bytes, dtype=np.float32)

print("\n--- EXTRACTED BINARY DATA ---")
print("Total data points found:", len(binary_data))
print("First 10 data points:", binary_data[:10])


FileNotFoundError: [Errno 2] No such file or directory: 'your_file.spe'

### line search (unreliable if binary and text at the same line)

In [ ]:
file_path = Path("your_file.spe")

# 1. Read the text header to find where the binary data starts
header_lines = []
binary_offset = 0

with open(file_path, "rb") as f:
    while True:
        line = f.readline()
        binary_offset += len(line)
        
        # Convert bytes to string to check for the end of the header
        try:
            line_str = line.decode('utf-8').strip()
        except UnicodeDecodeError:
            # If it fails to decode, we have officially hit the binary data block
            binary_offset -= len(line)
            break
            
        header_lines.append(line_str)
        
        # Common text indicators that the header has ended:
        if "End_of_Header" in line_str or "EOH" in line_str:
            break

print("--- HEADER METADATA ---")
print("\n".join(header_lines[:15])) # Print the first 15 lines of metadata

# 2. Read the binary array using NumPy
# (Adjust 'float32', 'float64', or 'int32' depending on what your header states)
with open(file_path, "rb") as f:
    f.seek(binary_offset) # Jump straight past the text header
    binary_data = np.fromfile(f, dtype=np.float32) 

print("\n--- EXTRACTED BINARY DATA ---")
print("Total data points found:", len(binary_data))
print("First 10 intensity counts:", binary_data[:10])







# Manual VMS transformation

In [ ]:
# standard usage:
# 1. set: perform_manual_tranform = True; dry_run = False
# 2. run the notebook, it will raise exception in this cell
# 3. do the spe to vms transformation by hand in the spe folder
# 4. run from this cell again, done
#
# omitting data copy but sitll getting path info stored in df_meat after the standard usage:
# 1. set: perform_manual_tranform = True; dry_run = True
# 2. the script will get you the vms/spe path without copying files
perform_manual_tranform = True
dry_run = False # you can use this to get related paths without wasting time on copying files

# copy and rename the spe files
if perform_manual_tranform:
    path_data_spe = path_data/'data_spe'
    df_md = read_and_write.copy_and_rename_spe(df_md, path_data_spe, dry_run)

# remind the user to do manual transform and copy vms files
if perform_manual_tranform:
    # raise exception to remind transformation
    if len(list(path_data_spe.glob('*.vms'))) == 0:
        print('No vms files found in the spe folder!')
        print('If this is the first run, do the tranform and run from this cell again.')
        raise Exception('No vms files found. Pls complete manual transformation and run again.')
    # copy the vms files
    path_data_vms = path_data/'data_vms'
    df_md = read_and_write.copy_and_rename_vms(df_md, path_data_vms, dry_run)
    df_md.to_csv(path_data/'df_meta_data.csv')

        


Missing 0 vms file(s)
Case used: Lower=False, Upper=True


In [ ]:
# path_script = path_script / 'example_data' / 'H1'
# path_script
# file_path = Path(r'RawData_USERNAME_2026-05-29_09384401\NTA000004.0H\NTA000004.23_20260517-152621\G94.NXB083468.01.01.20260517.152441.spe'.replace('\\', '/'), )
# file_path = path_script / file_path
# file_path.parent.mkdir(parents=True, exist_ok=True)
# file_path.touch()
# file_path.parent